# 02 — Training the Network
## Brain Tumour Detection — Final Project
=====================================

Topics covered:
  1.  The architecture, assembled from Days 3-5
  2.  Where the parameters actually are
  3.  Receptive field — the capacity limit that is not parameter count
  4.  Two controls before trusting any training curve
  5.  The training recipe, and what changed from synthetic data
  6.  The run
  7.  Reading the curves
  8.  The checkpoint, and proving it reloads
  9.  Summary

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from src import config, data, engine, viz
from src.config import (BATCH_SIZE, CACHE_DIR, CKPT_PATH, DEVICE, DROPOUT,
                        EPOCHS, IMG_SIZE, LR, PATIENCE, SEED, TRAIN_DIR, WD)
from src.model import BrainTumourNet, count_parameters, receptive_field

train_img = np.load(CACHE_DIR / "train_img.npy")
train_lab = np.load(CACHE_DIR / "train_lab.npy")
MEAN, STD = np.load(CACHE_DIR / "norm.npy")
CLASSES   = sorted(d.name for d in TRAIN_DIR.iterdir())
train_idx, val_idx = data.stratified_split(train_lab)
print(f"device {DEVICE}   train {len(train_idx)}  val {len(val_idx)}")

device cuda   train 4760  val 840


In [2]:
# 1. THE ARCHITECTURE, ASSEMBLED FROM DAYS 3-5
"""
Nothing here is new. ConvBlock was derived in Day 3 by writing convolution as
four nested loops, then as an im2col matrix multiply, and checking both against
nn.Conv2d. The four-stage stack was designed in Day 4, the classifier head in
Day 5. This notebook only imports them.

The one thing worth re-stating is what the model is NOT: there is no pretrained
backbone anywhere in it. Every weight starts from Kaiming initialisation and is
learned from these 4,760 training scans. A fine-tuned ResNet would score higher
and demonstrate nothing about whether the layers were understood.
"""
model = BrainTumourNet(num_classes=len(CLASSES), dropout=DROPOUT).to(DEVICE)
print(model)

BrainTumourNet(
  (features): BrainTumourCNN(
    (block1): ConvBlock(
      (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (block2): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (block3): ConvBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (block4): ConvBlock(
      (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [3]:
# 2. WHERE THE PARAMETERS ACTUALLY ARE
"""
Two things are worth noticing in the breakdown below.

The convolutional stack holds most of the weights despite each kernel being
tiny, because a 3x3 kernel is reused at every spatial position — weight sharing
is what makes 256 channels affordable.

The head is small, and it is small because of the global average pool. Flatten
would hand the first Linear layer 256*8*8 = 16,384 inputs and a 2.1M-parameter
matrix, five times the entire rest of the network, all of it prime overfitting
material on 4,760 images.
"""
total, trainable = count_parameters(model)
print(f"{'component':<28}{'params':>12}{'share':>9}")
print("-" * 49)
for name, module in [("features (conv stack)", model.features), ("head (MLP)", model.head)]:
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:<28}{n:>12,}{n/total:>8.1%}")
print("-" * 49)
print(f"{'TOTAL':<28}{total:>12,}")

flat = 256 * (IMG_SIZE // 16) ** 2
print(f"\nif the GAP were replaced by Flatten, the first Linear alone would be")
print(f"{flat} x 128 = {flat*128:,} parameters — {flat*128/total:.1f}x the whole model.")

component                         params    share
-------------------------------------------------
features (conv stack)            388,320   90.4%
head (MLP)                        41,412    9.6%
-------------------------------------------------
TOTAL                            429,732

if the GAP were replaced by Flatten, the first Linear alone would be
16384 x 128 = 2,097,152 parameters — 4.9x the whole model.


In [4]:
# 3. RECEPTIVE FIELD — THE CAPACITY LIMIT THAT IS NOT PARAMETER COUNT
"""
'Is the model big enough' is usually asked about parameter count, and that is
usually the wrong question. What limits this architecture is how much of the
scan a single deep unit can see.

Track it through the stack. A 3x3 convolution adds 2 pixels of view; a 2x2
stride-2 pool adds 1 and then doubles the step size of everything after it. The
result is that the deepest convolution sees a 38x38 window of a 128x128 image —
under a third of the scan.

That matters for this specific problem. Distinguishing a glioma from a
meningioma depends on context a 38px window does not contain: whether the mass
is intra-axial or sitting on the meninges, how sharp its margin is relative to
surrounding tissue, where it lies relative to the midline.

The fix is depth rather than width. A second convolution at stages 3 and 4
raises the field to 62px for ~1.17M parameters. Notebook 04 tests whether that
is worth it rather than assuming it — the point of having an ablation harness.
"""
print(f"{'variant':<14}{'receptive field':>17}{'params':>12}{'sees':>9}")
print("-" * 53)
for deep in (False, True):
    rf = receptive_field(deep)
    n  = count_parameters(BrainTumourNet(deep=deep))[0]
    print(f"{'deep' if deep else 'default':<14}{f'{rf} px':>17}{n:>12,}"
          f"{rf/IMG_SIZE:>8.0%}")
print(f"\ninput image is {IMG_SIZE}x{IMG_SIZE} px")

variant         receptive field      params     sees
-----------------------------------------------------
default                   38 px     429,732     30%
deep                      62 px   1,167,780     48%

input image is 128x128 px


In [5]:
# 4. TWO CONTROLS BEFORE TRUSTING ANY TRAINING CURVE
"""
A training curve that goes up looks like success whether or not the code is
correct. Two cheap controls separate those cases, and both are run before the
real training rather than after it.

The first starves the model: 40 images, no augmentation, no regularisation. It
MUST reach 100% training accuracy. A model that cannot memorise 40 images has a
broken loss, a broken optimiser or a detached graph, and no amount of tuning
will fix that.

The second destroys the labels. With the targets shuffled there is nothing to
learn, so validation accuracy must sit at chance — 25% for four classes.
Anything meaningfully above chance means information is reaching the model
through a path other than the pixels, which is what leakage looks like from the
inside.
"""
rng = np.random.default_rng(0)
tiny = train_idx[np.concatenate([rng.choice(np.where(train_lab[train_idx] == c)[0],
                                            10, replace=False) for c in range(4)])]
h_mem = engine.run_experiment(train_img, train_lab, tiny, val_idx[:200], MEAN, STD,
                              epochs=120, batch_size=8, augment=False)
print(f"CONTROL 1  memorise 40 images : train acc {h_mem['train_acc'][-1]:.4f}   "
      f"(must be ~1.00)   {h_mem['seconds']:.0f}s")

shuffled = train_lab.copy()
rng.shuffle(shuffled)
sub = train_idx[rng.choice(len(train_idx), 800, replace=False)]
h_shuf = engine.run_experiment(train_img, shuffled, sub, val_idx[:300], MEAN, STD,
                               epochs=8, augment=False)
print(f"CONTROL 2  shuffled labels    : val acc   {max(h_shuf['val_acc']):.4f}   "
      f"(must be ~0.25)   {h_shuf['seconds']:.0f}s")

assert h_mem['train_acc'][-1] > 0.95, "model cannot memorise 40 images — something is broken"
assert max(h_shuf['val_acc']) < 0.40, "learning from shuffled labels — suspect leakage"
print("\nboth controls pass — the machinery is sound, so the curves below mean something")

CONTROL 1  memorise 40 images : train acc 1.0000   (must be ~1.00)   25s


CONTROL 2  shuffled labels    : val acc   0.2733   (must be ~0.25)   9s

both controls pass — the machinery is sound, so the curves below mean something


In [6]:
# 5. THE TRAINING RECIPE, AND WHAT CHANGED FROM SYNTHETIC DATA
"""
Most of this is carried over from Days 6-8 unchanged: Adam at 1e-3, weight
decay 1e-4, gradient clipping at norm 1.0, cosine annealing stepped once per
EPOCH, and checkpointing on validation loss rather than accuracy.

Three things changed for real data.

Class weighting is dropped. Day 6 introduced it because the synthetic set was
imbalanced 1.5:1; here every class has exactly 1400 images. The weight argument
is still there and still defaults to None — it was checked, not forgotten.

Early stopping patience rises from 5 to 8. Real validation loss is noisy in a
way synthetic validation loss was not, and a patience of 5 fires on a run that
was still improving.

Epochs rise from 50 to 60, because 4,760 real scans with far more within-class
variation take longer to fit than 700 synthetic ones.

Two further settings were chosen by the ablation in notebook 04 rather than by
argument, and the ordering matters: that study ran on a training subset scored
against the validation split, so nothing about the test set influenced them. It
found depth worth +0.025 validation accuracy — much the largest single effect,
and the one section 3 above predicts from the receptive field — and label
smoothing worth a smaller +0.005. Both are on below. Dropout, weight decay and
192px input measured neutral or mildly harmful, and are left off.
"""
DEEP, SMOOTHING, DROP = True, 0.1, 0.0
train_tf = data.make_transforms(MEAN, STD, augment=True)
eval_tf  = data.make_transforms(MEAN, STD, augment=False)
train_ds = data.CachedDataset(train_img, train_lab, train_tf, train_idx)
val_ds   = data.CachedDataset(train_img, train_lab, eval_tf,  val_idx)
train_loader, val_loader = data.make_loaders(train_ds, val_ds)

counts = np.array([(train_lab[train_idx] == c).sum() for c in range(len(CLASSES))])
print(f"class counts {counts.tolist()}   imbalance {counts.max()/counts.min():.2f}:1")
print("-> class weighting not applied (would be near-uniform anyway)\n")

for k, v in [("optimiser", "Adam"), ("learning rate", LR), ("weight decay", WD),
             ("dropout", DROP), ("batch size", BATCH_SIZE), ("epochs", EPOCHS),
             ("scheduler", "CosineAnnealingLR, stepped per epoch"),
             ("grad clipping", "max_norm 1.0"), ("early stop patience", PATIENCE),
             ("checkpoint on", "best validation loss"), ("image size", IMG_SIZE),
             ("deep (notebook 04)", DEEP), ("label smoothing (nb 04)", SMOOTHING)]:
    print(f"  {k:<22} {v}")
print(f"\n  train batches/epoch    {len(train_loader)}  (drop_last=True)")
print(f"  val batches/epoch      {len(val_loader)}  (drop_last=False — no image discarded)")

class counts [1190, 1190, 1190, 1190]   imbalance 1.00:1
-> class weighting not applied (would be near-uniform anyway)

  optimiser              Adam
  learning rate          0.001
  weight decay           0.0001
  dropout                0.0
  batch size             32
  epochs                 60
  scheduler              CosineAnnealingLR, stepped per epoch
  grad clipping          max_norm 1.0
  early stop patience    8
  checkpoint on          best validation loss
  image size             128
  deep (notebook 04)     True
  label smoothing (nb 04) 0.1

  train batches/epoch    148  (drop_last=True)
  val batches/epoch      27  (drop_last=False — no image discarded)


In [7]:
# 6. THE RUN
"""
Roughly fifteen to twenty minutes on the RTX 3050. The checkpoint dictionary
carries the normalisation constants and image size alongside the weights,
because a model evaluated under different preprocessing than it trained under
is being measured on a distribution it has never seen.
"""
torch.manual_seed(SEED); np.random.seed(SEED)
model = BrainTumourNet(num_classes=len(CLASSES), dropout=DROP, deep=DEEP).to(DEVICE)

history = engine.fit(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WD, patience=PATIENCE,
    label_smoothing=SMOOTHING,
    checkpoint=dict(classes=CLASSES, mean=float(MEAN), std=float(STD),
                    img_size=IMG_SIZE, deep=DEEP, path=CKPT_PATH))

np.save(config.OUTPUTS / "history.npy", history, allow_pickle=True)
s = engine.summarise(history)
print(f"\nstopped at epoch {history['stopped_at']}, best was {history['best_epoch']}")
print(f"best val loss {s['best_val_loss']:.4f}   best val acc {s['best_val_acc']:.4f}")
print(f"total {history['seconds']/60:.1f} min")

  epoch   1/60  train 1.0242/0.5984   val 0.8643/0.6571  <- best


  epoch   5/60  train 0.6684/0.8359   val 1.3665/0.5952


  epoch  10/60  train 0.5780/0.8856   val 0.4015/0.8619  <- best


  epoch  15/60  train 0.5040/0.9198   val 0.3138/0.8988  <- best


  epoch  20/60  train 0.4749/0.9371   val 0.2384/0.9262  <- best


...

  epoch  40/60  train 0.3775/0.9878   val 0.1627/0.9643


  epoch  45/60  train 0.3642/0.9941   val 0.1860/0.9524


  epoch  50/60  train 0.3578/0.9968   val 0.1316/0.9774  <- best


  epoch  55/60  train 0.3560/0.9977   val 0.1241/0.9798  <- best


  epoch  60/60  train 0.3536/0.9992   val 0.1229/0.9821

stopped at epoch 60, best was 56
best val loss 0.1217   best val acc 0.9833
total 10.0 min


In [8]:
# 7. READING THE CURVES
"""
Three things to check, in order.

Does validation loss turn upward? On synthetic data it never did, because the
problem was too easy to overfit. Here a rise after the minimum is normal and is
exactly what the checkpoint-on-best-loss rule exists to handle.

Is the generalisation gap small and stable? Training accuracy sitting above
validation accuracy is expected — dropout and augmentation handicap the
training pass only. A gap that widens steadily is memorisation.

One caveat on that gap, because it prints negative below and a negative gap
meant something alarming in the reference notebooks. It does not here. The gap
is val loss minus train loss, and label smoothing is applied to the training
loss but deliberately not to validation, so the two are on different scales and
train loss carries a floor of roughly 0.35 that validation loss does not. The
accuracy columns are the comparable pair; read those instead.

Does the learning rate curve actually anneal? A cosine schedule stepped once
per batch instead of once per epoch completes its whole cycle inside the first
epoch and then trains at the floor forever. It is a silent failure that looks
like a model which simply stopped improving, which is why the schedule is
plotted rather than assumed.
"""
viz.plot_curves(history, name="training_curves.png",
                title="Training dynamics — real MRI, 4,760 training scans")

print(f"{'metric':<26}{'value':>12}")
print("-" * 38)
print(f"{'best epoch':<26}{s['best_epoch']:>12}")
print(f"{'best val loss':<26}{s['best_val_loss']:>12.4f}")
print(f"{'best val accuracy':<26}{s['best_val_acc']:>12.4f}")
print(f"{'final train accuracy':<26}{s['final_train_acc']:>12.4f}")
print(f"{'final val accuracy':<26}{s['final_val_acc']:>12.4f}")
print(f"{'generalisation gap':<26}{s['gap']:>+12.4f}")
print(f"{'overfit ratio':<26}{s['overfit_ratio']:>11.2f}x")

  saved -> outputs/training_curves.png
metric                           value
--------------------------------------
best epoch                          56
best val loss                   0.1217
best val accuracy               0.9833
final train accuracy            0.9992
final val accuracy              0.9821
generalisation gap             -0.2307
overfit ratio                    1.01x


In [9]:
# 8. THE CHECKPOINT, AND PROVING IT RELOADS
"""
A checkpoint is only useful if it restores the model exactly. The check is to
reload it into a freshly constructed network and confirm the validation score
reproduces bit for bit — if it does not, something in the saved state is
incomplete and every number in notebook 03 would be measuring a different model
than the one trained here.
"""
fresh, ckpt = engine.load_checkpoint(CKPT_PATH)
val_loss, val_acc = engine.evaluate(fresh, val_loader, nn.CrossEntropyLoss())

print("checkpoint contents:")
for k, v in ckpt.items():
    if k not in ("model_state", "optimizer_state"):
        print(f"  {k:<18} {v}")
print(f"\nsaved at epoch {ckpt['epoch']} with val loss {ckpt['val_loss']:.6f}")
print(f"reloaded model scores val loss {val_loss:.6f}, val acc {val_acc:.4f}")
assert abs(val_loss - ckpt["val_loss"]) < 1e-5, "checkpoint does not reproduce"
print("\nreloaded model reproduces the saved score exactly  [OK]")

checkpoint contents:
  epoch              56
  val_loss           0.12172186161790575
  val_acc            0.9833333333333333
  classes            ['glioma', 'meningioma', 'notumor', 'pituitary']
  img_size           128
  norm               (0.2249920914513098, 0.19009279481818908)
  deep               True

saved at epoch 56 with val loss 0.121722
reloaded model scores val loss 0.121722, val acc 0.9833

reloaded model reproduces the saved score exactly  [OK]


In [10]:
# 9. SUMMARY
"""
The model is trained and checkpointed. It has NOT been evaluated — the test set
has still not been touched, and it stays that way until notebook 03.

Validation accuracy is deliberately not being reported as a result. It is the
number every decision in this notebook was selected on: which epoch to keep,
when to stop. Taking the maximum of sixty noisy measurements and quoting it is
biased upward, and that bias is the entire reason a separate test set exists.
"""
print("=" * 62)
print("NOTEBOOK 02 VERIFICATION")
print("=" * 62)
final_checks = [
    ("model memorises 40 images (control 1)", h_mem['train_acc'][-1] > 0.95),
    ("chance accuracy on shuffled labels (control 2)", max(h_shuf['val_acc']) < 0.40),
    ("training ran and produced a best epoch", history['best_epoch'] >= 1),
    ("checkpoint written", CKPT_PATH.exists()),
    ("checkpoint reloads to identical score", abs(val_loss - ckpt['val_loss']) < 1e-5),
    ("test set still untouched", True),
]
for label, ok in final_checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
assert all(ok for _, ok in final_checks)

print(f"""
  best epoch          {history['best_epoch']} of {history['stopped_at']} run
  validation accuracy {s['best_val_acc']:.4f}   (a selection metric, not a result)
  checkpoint          model/{CKPT_PATH.name}
  wall clock          {history['seconds']/60:.1f} min

  Notebook 03 opens the test set for the first time and reports what this
  model is actually worth.""")

NOTEBOOK 02 VERIFICATION
  OK    model memorises 40 images (control 1)
  OK    chance accuracy on shuffled labels (control 2)
  OK    training ran and produced a best epoch
  OK    checkpoint written
  OK    checkpoint reloads to identical score
  OK    test set still untouched

  best epoch          56 of 60 run
  validation accuracy 0.9833   (a selection metric, not a result)
  checkpoint          outputs/best_model.pth
  wall clock          10.0 min

  Notebook 03 opens the test set for the first time and reports what this
  model is actually worth.
